# Curso 2 — Redes neuronales### De una neurona a una red convolucional, sobre el mismo problema---En el notebook anterior entrenamos una regresión logística sobre los píxeles crudosy anduvo mal. No fue mala suerte: era **inevitable**, y entender por qué es la mitadde esta clase.Acá vamos a construir cuatro modelos sobre exactamente los mismos datos, y a medirloscon exactamente las mismas métricas. Lo único que cambia es la arquitectura.| Modelo | Idea | Qué esperar ||---|---|---|| Regresión logística | una sola neurona | el piso || Red densa (MLP) | muchas neuronas apiladas | mejora poco || Red convolucional (CNN) | mira vecindarios de píxeles | mejora mucho || Transfer learning | una red que ya vio millones de fotos | el techo |> **GPU obligatoria acá.** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU`.> Sin GPU el último modelo puede tardar una hora en lugar de cinco minutos.

---## 0. Preparar

In [ ]:
import sys, osfrom pathlib import Pathdef en_colab():    try:        import google.colab  # noqa        return True    except ImportError:        return Falseif en_colab():    CARPETA = "Deteccion-De-Fracturas-ETRR"    if not Path(CARPETA).exists():        !git clone -q https://github.com/resagri-fiuba/Deteccion-De-Fracturas-ETRR.git    RAIZ = Path("/content") / CARPETAelse:    RAIZ = Path.cwd()    while not (RAIZ / "src").exists() and RAIZ != RAIZ.parent:        RAIZ = RAIZ.parentsys.path.insert(0, str(RAIZ)); os.chdir(RAIZ)import numpy as np, pandas as pd, torchimport matplotlib.pyplot as pltfrom torch import nnfrom sklearn.model_selection import train_test_splitfrom src import cursotorch.manual_seed(42); np.random.seed(42)DISP = curso.dispositivo()

In [ ]:
# Los mismos datos del notebook anterior, misma partición, misma semilla.X_ent, y_ent, X_test, y_test = curso.cargar_cifar10("datos")y_ent_bin  = curso.a_binario(y_ent)y_test_bin = curso.a_binario(y_test)X_tr, X_val, y_tr, y_val = train_test_split(    X_ent, y_ent_bin, test_size=0.20, stratify=y_ent_bin, random_state=42)print(f"entrenamiento {len(y_tr):,}  ·  validación {len(y_val):,}  ·  test {len(y_test_bin):,}")print(f"gatos: {y_tr.mean():.1%}")

---## 1. Qué es, en concreto, una neuronaUna neurona hace tres cosas, y nada más:1. Multiplica cada entrada por un **peso**2. Suma todo, más un valor fijo llamado **sesgo**3. Pasa el resultado por una **función de activación** que decide cuánto deja pasarEn fórmula: `salida = activación(w₁x₁ + w₂x₂ + ... + wₙxₙ + b)`**La regresión logística del notebook anterior era exactamente esto**, con la funciónsigmoide como activación. Una sola neurona con 3.072 entradas.Una **red** es lo que pasa cuando apilás neuronas en capas: la salida de cada capa esla entrada de la siguiente. Y el motivo por el que eso sirve es la activación: sinella, apilar capas lineales da otra capa lineal y no ganás nada. La activación es loque permite representar relaciones curvas, no rectas.

---## 2. Los datos como tensoresPyTorch no trabaja con arrays de numpy sino con **tensores**, que son lo mismo perocon dos capacidades extra: pueden vivir en la GPU y recuerdan las operaciones que seles hicieron, para poder derivarlas después.Dos conversiones obligatorias:- **Reordenar los ejes.** Numpy y las imágenes usan `(n, alto, ancho, canales)`;  PyTorch quiere `(n, canales, alto, ancho)`. Confundir esto es el error número uno.- **Normalizar** de 0–255 a 0–1.El **DataLoader** además parte los datos en **lotes** (*batches*). No se entrena conlas 40.000 imágenes de una: no entran en memoria y además conviene corregir los pesosseguido, no una vez por vuelta.

In [ ]:
dl_tr   = curso.a_tensores(X_tr,   y_tr,   tamano_lote=128, mezclar=True)dl_val  = curso.a_tensores(X_val,  y_val,  tamano_lote=256)dl_test = curso.a_tensores(X_test, y_test_bin, tamano_lote=256)xb, yb = next(iter(dl_tr))print(f"Un lote de imágenes : {tuple(xb.shape)}  →  (lote, canales, alto, ancho)")print(f"Un lote de etiquetas: {tuple(yb.shape)}")print(f"Rango de valores    : {xb.min():.2f} a {xb.max():.2f}")print(f"Lotes por época     : {len(dl_tr)}")

---## 3. El bucle de entrenamiento, a manoEsto es **el corazón de todo el aprendizaje profundo**. Son cinco pasos que se repitenmillones de veces, y no hay nada más. Vale la pena escribirlo a mano una vez.```para cada lote:    1. predecir            salida = modelo(x)    2. medir el error      perdida = criterio(salida, y)    3. borrar lo anterior  optimizador.zero_grad()    4. calcular la culpa   perdida.backward()    5. corregir            optimizador.step()```El paso 4 es la retropropagación: PyTorch recorre la red hacia atrás y calcula, paracada peso, **cuánto contribuyó al error**. El paso 5 mueve cada peso un poquito en ladirección que reduce ese error. El "poquito" es la tasa de aprendizaje (`lr`).### Un detalle que en nuestro problema es decisivoCon 10% de positivos, el camino más fácil para la red es decir "no" a todo: la pérdidabaja rápido y no aprende nada. `pos_weight` le dice al criterio que equivocarse con ungato cuesta 9 veces más que equivocarse con un no-gato. Es la forma más simple decompensar el desbalance, y **es igual de necesaria con las fracturas**.

In [ ]:
# Primer modelo: una red densa (MLP). Aplana la imagen y apila dos capas ocultas.mlp = nn.Sequential(    nn.Flatten(),               # (n,3,32,32) → (n,3072)    nn.Linear(3072, 256),       # capa oculta    nn.ReLU(),                  # activación: deja pasar lo positivo, corta lo negativo    nn.Dropout(0.3),            # apaga el 30% de las neuronas al azar: evita memorizar    nn.Linear(256, 64),    nn.ReLU(),    nn.Linear(64, 1),           # una sola salida: el score de "es gato")print(mlp)print(f"\nParámetros a aprender: {sum(p.numel() for p in mlp.parameters()):,}")

In [ ]:
# El bucle escrito a mano, para verlo una vez completo.peso_pos = torch.tensor([(1 - y_tr.mean()) / y_tr.mean()], device=DISP)print(f"pos_weight = {peso_pos.item():.1f}  (un gato pesa {peso_pos.item():.0f} veces más)")mlp = mlp.to(DISP)criterio = nn.BCEWithLogitsLoss(pos_weight=peso_pos)optimizador = torch.optim.Adam(mlp.parameters(), lr=1e-3)hist_mlp = {"ent": [], "val": []}for epoca in range(1, 9):    mlp.train()    suma = 0.0    for xb, yb in dl_tr:        xb, yb = xb.to(DISP), yb.to(DISP)        salida  = mlp(xb)                    # 1. predecir        perdida = criterio(salida, yb)       # 2. medir el error        optimizador.zero_grad()              # 3. borrar gradientes viejos        perdida.backward()                   # 4. repartir la culpa        optimizador.step()                   # 5. corregir los pesos        suma += perdida.item() * len(xb)    hist_mlp["ent"].append(suma / len(dl_tr.dataset))    mlp.eval()                               # apaga dropout    suma = 0.0    with torch.no_grad():                    # sin calcular gradientes: más rápido        for xb, yb in dl_val:            xb, yb = xb.to(DISP), yb.to(DISP)            suma += criterio(mlp(xb), yb).item() * len(xb)    hist_mlp["val"].append(suma / len(dl_val.dataset))    print(f"época {epoca}/8   entrenamiento {hist_mlp['ent'][-1]:.4f}   "          f"validación {hist_mlp['val'][-1]:.4f}")

In [ ]:
curso.curva_entrenamiento(hist_mlp, "Red densa (MLP)");

### Cómo se lee esta curva- **Las dos bajan juntas** → el modelo está aprendiendo. Todo bien.- **La de entrenamiento baja y la de validación sube** → *sobreajuste*: el modelo  dejó de aprender y empezó a memorizar. Hay que parar ahí.- **Ninguna baja** → algo está roto: tasa de aprendizaje mal, datos mal, o el  problema no tiene señal.

In [ ]:
score_mlp, real_val = curso.predecir(mlp, dl_val, DISP)umbral_mlp, _, _ = curso.umbral_para_recall(real_val, score_mlp, 0.90)m_mlp = curso.informe(real_val, (score_mlp >= umbral_mlp).astype(int), score_mlp, "MLP")

---## 4. Por qué una CNN ganaMirá la primera línea del MLP: `nn.Flatten()`.Ahí se rompe todo. Aplanar convierte la imagen en una lista de 3.072 números sueltos y**destruye la información de qué píxel está al lado de cuál**. Para el MLP, dos píxelesvecinos no tienen ninguna relación especial; son las columnas 500 y 501 de una tabla.Una **red convolucional** no aplana. Desliza filtros chiquitos (3×3) por toda la imagenbuscando patrones locales: bordes, esquinas, texturas. Y como el mismo filtro recorretoda la imagen, aprende que **un borde es un borde esté donde esté**.Tres consecuencias:1. Muchísimos menos parámetros: un filtro 3×3 tiene 9 pesos, no 3.072.2. Aprende jerarquías: bordes → texturas → partes → objetos.3. Es lo que se usa en imágenes médicas, por exactamente los mismos motivos.

In [ ]:
cnn = nn.Sequential(    # Bloque 1: 32x32 → 16x16    nn.Conv2d(3, 32, kernel_size=3, padding=1),   # 32 filtros de 3x3    nn.BatchNorm2d(32),                            # estabiliza el entrenamiento    nn.ReLU(),    nn.Conv2d(32, 32, kernel_size=3, padding=1),    nn.BatchNorm2d(32), nn.ReLU(),    nn.MaxPool2d(2),                               # se queda con el máximo de cada 2x2    # Bloque 2: 16x16 → 8x8    nn.Conv2d(32, 64, kernel_size=3, padding=1),    nn.BatchNorm2d(64), nn.ReLU(),    nn.Conv2d(64, 64, kernel_size=3, padding=1),    nn.BatchNorm2d(64), nn.ReLU(),    nn.MaxPool2d(2),    # Bloque 3: 8x8 → 4x4    nn.Conv2d(64, 128, kernel_size=3, padding=1),    nn.BatchNorm2d(128), nn.ReLU(),    nn.MaxPool2d(2),    # Cabeza: de mapas de características a una decisión    nn.AdaptiveAvgPool2d(1),    nn.Flatten(),    nn.Dropout(0.3),    nn.Linear(128, 1),)print(f"Parámetros del MLP: {sum(p.numel() for p in mlp.parameters()):,}")print(f"Parámetros de la CNN: {sum(p.numel() for p in cnn.parameters()):,}")print("\nLa CNN tiene MENOS parámetros y va a andar bastante mejor.")print("Eso es lo que significa 'tener la arquitectura adecuada al problema'.")

In [ ]:
# A partir de acá usamos la función empaquetada: es el mismo bucle de recién.hist_cnn = curso.entrenar(cnn, dl_tr, dl_val, epocas=10, lr=1e-3,                          peso_positivos=float(peso_pos), disp=DISP)curso.curva_entrenamiento(hist_cnn, "Red convolucional");

In [ ]:
score_cnn, _ = curso.predecir(cnn, dl_val, DISP)umbral_cnn, _, _ = curso.umbral_para_recall(real_val, score_cnn, 0.90)m_cnn = curso.informe(real_val, (score_cnn >= umbral_cnn).astype(int), score_cnn, "CNN")

---## 5. Transfer learning: no empezar de cero**ResNet18** es una red que ya fue entrenada con más de un millón de fotos deImageNet. En el camino aprendió a detectar bordes, texturas, formas y partes deobjetos — cosas que sirven para *cualquier* problema de imágenes, no solo para elque la entrenaron.La idea del transfer learning: tomamos esa red, le cambiamos la última capa por unaque responda nuestra pregunta, y la reentrenamos un poco con nuestros datos.**Es lo que vamos a hacer con las radiografías**, y es lo que hace que un proyectoescolar con 4.000 imágenes pueda dar resultados decentes. Entrenar desde ceronecesitaría cien veces más datos.> Nadie entrena una red grande desde cero. Ni en la escuela, ni en la industria.

In [ ]:
from torchvision import modelsimport torch.nn.functional as Fresnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)# La última capa de ResNet18 predice 1.000 clases de ImageNet.# La reemplazamos por una que predice una sola cosa: ¿es gato?print(f"Última capa original : {resnet.fc}")resnet.fc = nn.Linear(resnet.fc.in_features, 1)print(f"Última capa nueva    : {resnet.fc}")# ResNet espera imágenes más grandes que 32x32. Las agrandamos al vuelo.class ConEscala(nn.Module):    def __init__(self, red, tamano=96):        super().__init__()        self.red, self.tamano = red, tamano    def forward(self, x):        x = F.interpolate(x, size=self.tamano, mode="bilinear", align_corners=False)        return self.red(x)resnet = ConEscala(resnet, tamano=96)print(f"\nParámetros: {sum(p.numel() for p in resnet.parameters()):,}")

In [ ]:
# Tasa de aprendizaje MÁS BAJA que antes: no queremos destruir lo que ya sabe.hist_rn = curso.entrenar(resnet, dl_tr, dl_val, epocas=4, lr=3e-4,                         peso_positivos=float(peso_pos), disp=DISP)curso.curva_entrenamiento(hist_rn, "ResNet18 preentrenada");

In [ ]:
score_rn, _ = curso.predecir(resnet, dl_val, DISP)umbral_rn, _, _ = curso.umbral_para_recall(real_val, score_rn, 0.90)m_rn = curso.informe(real_val, (score_rn >= umbral_rn).astype(int), score_rn, "ResNet18")

---## 6. Los cuatro modelos, uno al lado del otroEsta tabla es el entregable del notebook. Cada fila costó unos minutos de GPU yla diferencia entre ellas no es el poder de cómputo: es **la arquitectura**.

In [ ]:
tabla = pd.DataFrame([m_mlp, m_cnn, m_rn]).set_index("modelo")tabla = tabla[["recall", "precision", "f1", "roc_auc", "pr_auc", "accuracy"]]tabla.round(3)

In [ ]:
# Las tres curvas precisión-recall superpuestas: se ve la mejora de un vistazofrom sklearn.metrics import precision_recall_curve, average_precision_scorefig, eje = plt.subplots(figsize=(6.4, 4.6))for nombre, sc, color in [("MLP", score_mlp, curso.GRIS),                          ("CNN", score_cnn, curso.AZUL),                          ("ResNet18", score_rn, curso.ROJO)]:    p, r, _ = precision_recall_curve(real_val, sc)    eje.plot(r, p, label=f"{nombre}  (AUC {average_precision_score(real_val, sc):.3f})",             linewidth=2, color=color)eje.axhline(real_val.mean(), color="#999", linestyle="--", linewidth=1)eje.text(.02, real_val.mean() + .02, "azar", color="#999", fontsize=9)eje.set_xlabel("recall — de los gatos reales, cuántos encuentra")eje.set_ylabel("precisión — cuando dice gato, cuántas veces acierta")eje.set_title("Los tres modelos, todos los umbrales a la vez", fontsize=12)eje.legend(); eje.spines[["top", "right"]].set_visible(False)fig.tight_layout()

---## 7. Qué nos llevamos a las radiografíasEl plan de la semana 3 del proyecto es, línea por línea, lo que acabás de hacer:1. **Transfer learning, siempre.** ResNet18 preentrenada, última capa cambiada.   Con 4.000 radiografías es la única opción sensata.2. **`pos_weight` para el desbalance.** Con 17,6% de fracturas, el valor es ≈ 4,7.   Sin eso, el modelo aprende a decir "sano" a todo.3. **La curva de entrenamiento se mira siempre.** Es el primer diagnóstico   cuando algo no anda.4. **Umbral elegido por recall**, no 0,5 por costumbre.5. **La tabla comparativa es el entregable**, no el mejor número suelto.Lo único distinto va a ser el tamaño de las imágenes y que hay que partir por paciente.---## 🔧 Ejercicios1. **Fácil.** Sacá `pos_weight` (poné `peso_positivos=None`) y reentrená la CNN.   Mirá el recall. ¿Qué aprendió el modelo a hacer?2. **Fácil.** Subí la tasa de aprendizaje a `1e-1` y entrená 3 épocas.   ¿Qué le pasa a la curva de pérdida? Es importante ver esto una vez.3. **Media.** Agregá un bloque convolucional más a la CNN. ¿Mejora o empeora?   ¿Cuántos parámetros tiene ahora?4. **Media.** Congelá todas las capas de ResNet menos la última   (`for p in resnet.red.parameters(): p.requires_grad = False`, y volvé a habilitar `fc`).   Entrená. ¿Cuánto más rápido va? ¿Cuánto pierde en calidad?5. **Difícil.** Agregá aumento de datos: volteo horizontal y recortes al azar,   solo en entrenamiento. ¿Mejora la validación? Ojo con esto en radiografías:   voltear una muñeca izquierda la convierte en derecha.6. **Difícil.** Entrená la CNN con solo 2.000 imágenes en vez de 40.000 y compará   contra ResNet18 con esas mismas 2.000. La diferencia entre las dos es exactamente   el argumento a favor del transfer learning en proyectos con pocos datos.---**Siguiente:** aplicar todo esto a FracAtlas — `notebooks/02_particion.ipynb`y `notebooks/03_clasificador.ipynb`.